# P3 LIP — RepDWNet + WDice on Kaggle

One notebook: choose Early/Late/Middle/First3/All in the first config cell. Pipeline follows the successful WDice/DPP notebooks: clone source → detect attached PhenoBench → resolve config → tests and GPU smoke runs → GPU benchmark → full training → optional offline analysis → one ZIP.

Enable Internet and GPU T4 ×2. The source files must be committed and pushed to `f/down-lip` before Save & Run All. Recipe: 100 epochs, validation every 5, FP32, batch4/GPU; standalone Bottleneck LIP B=C/4 at the selected Down stages (Ex1–Ex5). No CUDA extension.

The benchmark is single-GPU synthetic compute, separate from the two-GPU smoke/full training. Offline Enrichment/Entropy is optional after training and can also be run later from the ZIP.


In [ ]:
# FIRST CONFIG CELL: choose the experiment and source here.
REPO_URL = 'https://github.com/FUAVX/RPD_Reproduce.git'
GIT_REF = 'f/down-lip'
SOURCE_COMMIT = ''  # optional full commit SHA reachable from GIT_REF
EXPERIMENT_MODE = 'ex4'  # ex1 Early; ex2 Late; ex3 Middle; ex4 First3; ex5 All
LIP_VARIANT = 'lip_bottleneck'  # fixed audited LIP-Now topology
RUN_TAG = 'run1'  # use a new tag for a separate repeat
DATASET_DIR = ''  # empty = auto-detect attached PhenoBench Kaggle input
NUM_WORKERS = 2
DEVICES = 2  # P2 runtime override; YAML default remains 1
WORKING_ROOT = '/kaggle/working'

EXPERIMENTS = {
    'ex1': ('P3_LIP_Ex1_Early', [1, 2]),
    'ex2': ('P3_LIP_Ex2_Late', [3, 4]),
    'ex3': ('P3_LIP_Ex3_Middle', [2, 3]),
    'ex4': ('P3_LIP_Ex4_First3', [1, 2, 3]),
    'ex5': ('P3_LIP_Ex5_All', [1, 2, 3, 4]),
}
if EXPERIMENT_MODE not in EXPERIMENTS:
    raise ValueError(f'Unknown EXPERIMENT_MODE: {EXPERIMENT_MODE}')
if LIP_VARIANT != 'lip_bottleneck':
    raise NotImplementedError(f'{LIP_VARIANT} is not implemented. Implement and test the variant before selecting it.')
if DEVICES != 2:
    raise ValueError('This P2-matched protocol requires 2 devices.')
SCENARIO, EXPECTED_STAGES = EXPERIMENTS[EXPERIMENT_MODE]

RUN_OFFLINE_ANALYSIS = False  # optional; may also run later using ZIP checkpoints
ANALYSIS_MAX_IMAGES = None  # None = all validation images; deterministic filename order


In [ ]:
# Kaggle runtime dependencies. Kaggle supplies torch/CUDA; do not reinstall torch.
!nvidia-smi
# Lightning 1.9.5 uses np.Inf; keep NumPy 1.x as specified in P2 requirements.
!pip install -q 'numpy<2' 'pytorch-lightning==1.9.5' 'torchmetrics==0.11.4' 'setuptools<70' tensorboard matplotlib pandas tabulate timm scikit-image tifffile albumentations pyyaml rich


In [ ]:
import json
import pathlib
import shutil
import subprocess
import yaml

WORKING_DIR = pathlib.Path(WORKING_ROOT)
REPO_DIR = WORKING_DIR / 'RPD_P2'
RUN_DIR = WORKING_DIR / f'{SCENARIO}_{LIP_VARIANT}_{RUN_TAG}'
if RUN_DIR.exists():
    raise FileExistsError(f'Output already exists: {RUN_DIR}. Choose a new RUN_TAG.')

def run(command, cwd=None):
    print('+', ' '.join(map(str, command)))
    subprocess.run(command, cwd=cwd, check=True)

if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)
run(['git', 'clone', '--branch', GIT_REF, '--single-branch', REPO_URL, str(REPO_DIR)])
if SOURCE_COMMIT:
    run(['git', 'checkout', '--detach', SOURCE_COMMIT], cwd=REPO_DIR)
run(['git', 'rev-parse', '--verify', 'HEAD'], cwd=REPO_DIR)

scenario_path = REPO_DIR / 'configs' / 'p3_scenarios' / f'{SCENARIO}.yaml'
required_paths = [scenario_path, REPO_DIR / 'test_wdice.py', REPO_DIR / 'test_lip.py',
                  REPO_DIR / 'test_lip_analysis.py', REPO_DIR / 'models/rpdnet/lip.py',
                  REPO_DIR / 'models/rpdnet/downsampling.py', REPO_DIR / 'scripts/benchmark_lip.py',
                  REPO_DIR / 'scripts/analyze_lip.py', REPO_DIR / 'test_config_callback.py',
                  REPO_DIR / 'test_tensorboard_logging.py', REPO_DIR / 'utils/tensorboard_compat.py']
missing = [str(path) for path in required_paths if not path.is_file()]
if missing:
    raise FileNotFoundError('Branch does not contain the committed P3 LIP files: ' + ', '.join(missing))

def is_phenobench_root(path):
    return (
        (path / 'train' / 'images').is_dir()
        and (path / 'train' / 'semantics').is_dir()
        and (path / 'val' / 'images').is_dir()
        and (path / 'val' / 'semantics').is_dir()
    )

if DATASET_DIR:
    dataset_dir = pathlib.Path(DATASET_DIR)
else:
    kaggle_input = pathlib.Path('/kaggle/input')
    candidates = [
        kaggle_input / 'datasets' / 'ndhoang2310' / 'phenobench-dataset' / 'PhenoBench',
        kaggle_input / 'phenobench-dataset' / 'PhenoBench',
        kaggle_input / 'datasets' / 'ndhoang2310' / 'phenobench-dataset',
        kaggle_input / 'phenobench-dataset',
    ]
    if kaggle_input.is_dir():
        attached_inputs = [p for p in kaggle_input.iterdir() if p.is_dir()]
        candidates.extend(p / 'PhenoBench' for p in attached_inputs)
        candidates.extend(attached_inputs)
    dataset_dir = next((p for p in candidates if is_phenobench_root(p)), None)
if dataset_dir is None or not is_phenobench_root(dataset_dir):
    raise FileNotFoundError('Cannot find a complete PhenoBench root in /kaggle/input. Attach the PhenoBench dataset, or set DATASET_DIR to its root containing train/images, train/semantics, val/images, and val/semantics.')

print(f'[OK] source ref: {subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO_DIR, text=True).strip()}')
print(f'[OK] dataset: {dataset_dir}')
print(f'[OK] train images: {len(list((dataset_dir / "train" / "images").glob("*")))}')


In [ ]:
# Resolve the Kaggle-specific configuration without modifying the cloned repository.
RUN_DIR.mkdir(parents=True, exist_ok=True)
with scenario_path.open(encoding='utf-8') as stream:
    cfg = yaml.safe_load(stream)

cfg['data']['path_to_dataset'] = str(dataset_dir)
cfg['data']['num_workers'] = NUM_WORKERS
active_cfg_path = RUN_DIR / f'active_config_{SCENARIO}.yaml'
with active_cfg_path.open('w', encoding='utf-8') as stream:
    yaml.safe_dump(cfg, stream, sort_keys=False)

assert cfg['train']['loss'] == 'weighted_dice'
assert cfg['loss']['dice']['lambda'] == 1.0
assert cfg['loss']['dice']['epsilon'] == 1e-6
assert cfg['train']['class_weights'] == [1.47, 5.06, 10.02]
assert cfg['train']['max_epoch'] == 100
assert cfg['val']['check_val_every_n_epoch'] == 5
assert cfg['train']['batch_size'] == 4
assert cfg['train']['early_stopping']['mode'] == 'none'
assert cfg['backbone']['downsampling']['variant'] == LIP_VARIANT
assert cfg['backbone']['downsampling']['stages'] == EXPECTED_STAGES
# Match against the source P2 config to catch unintended recipe drift.
import copy
base_cfg = yaml.safe_load((REPO_DIR / 'configs/p2_scenarios/P2_Ex7_RepDWNet_WDice-Loss.yaml').read_text())
base_cfg['data']['path_to_dataset'] = str(dataset_dir)
base_cfg['data']['num_workers'] = NUM_WORKERS
inherited = copy.deepcopy(cfg)
del inherited['backbone']['downsampling']
inherited['experiment'] = base_cfg['experiment']
assert inherited == base_cfg, 'P3 differs from P2 beyond experiment metadata/downsampling.'
print(yaml.safe_dump({
    'experiment': cfg['experiment'],
    'backbone': cfg['backbone'],
    'loss': cfg['loss'],
}, sort_keys=False))

assert cfg['train']['precision'] == 32
runtime_cfg = copy.deepcopy(cfg)
runtime_cfg['train']['devices'] = DEVICES
runtime_cfg['val']['devices'] = 1  # same CLI override behavior as WDice/DPP
runtime_cfg_path = RUN_DIR / 'runtime_config.yaml'
runtime_cfg_path.write_text(yaml.safe_dump(runtime_cfg, sort_keys=False), encoding='utf-8')


In [ ]:
# Actual hardware check; the full training and all placement smoke runs use both GPUs.
import torch
if not torch.cuda.is_available() or torch.cuda.device_count() != DEVICES:
    raise RuntimeError('Select GPU T4 x2 in Kaggle settings before running.')
print('GPUs:', [torch.cuda.get_device_name(i) for i in range(DEVICES)])
# Fast CPU regression tests, including real TensorBoard and the previous DDP config-export fix.
for script in ['test_tensorboard_logging.py', 'test_wdice.py', 'test_dpp.py',
               'test_lip.py', 'test_lip_analysis.py', 'test_config_callback.py']:
    run(['python', script], cwd=REPO_DIR)

# Small synthetic batches verify actual WDice trainer, checkpoints and 2-GPU DDP for every placement.
for mode, (scenario, stages) in EXPERIMENTS.items():
    run(['python', 'test_lip.py', '--smoke', '--config',
         str(REPO_DIR / 'configs/p3_scenarios' / f'{scenario}.yaml'),
         '--output-dir', str(RUN_DIR / 'smoke' / mode), '--accelerator', 'gpu',
         '--devices', str(DEVICES)], cwd=REPO_DIR)

# Required pre-campaign microbenchmark: Control vs selected LIP placement, FP32, batch4 at train resolution.
# Measures a single GPU; these figures do not claim DDP or dataloader throughput.
run(['python', 'scripts/benchmark_lip.py', '--device', 'cuda',
     '--lip-config', str(scenario_path),
     '--output-dir', str(RUN_DIR / 'benchmark')], cwd=REPO_DIR)

TRAIN_LOG_DIR = RUN_DIR / 'training_output'
train_command = [
    'python', 'multi_metric_train.py', '--config', str(active_cfg_path),
    '--export_dir', str(TRAIN_LOG_DIR), '--devices', str(DEVICES),
]
# Launch directly as in the successful P2 WDice notebook.
run(train_command, cwd=REPO_DIR)


In [ ]:
# Optional offline behavior analysis. No hooks or GT importance metrics run during training.
if RUN_OFFLINE_ANALYSIS:
    candidates = sorted((TRAIN_LOG_DIR / 'checkpoints').glob('*best_weedIoU*.ckpt'))
    if len(candidates) != 1:
        raise RuntimeError(f'Expected one best Weed-IoU checkpoint, found {candidates}')
    analysis_command = ['python', 'scripts/analyze_lip.py', '--config', str(runtime_cfg_path),
                        '--checkpoint', str(candidates[0]), '--dataset-dir', str(dataset_dir),
                        '--output-dir', str(RUN_DIR / 'lip_analysis'), '--device', 'cuda']
    if ANALYSIS_MAX_IMAGES is not None:
        analysis_command += ['--max-images', str(ANALYSIS_MAX_IMAGES)]
    run(analysis_command, cwd=REPO_DIR)

# Package artifacts after successful training, following the WDice/DPP flow.
import importlib.metadata
source_commit = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO_DIR, text=True).strip()
manifest = {
    'experiment': SCENARIO, 'mode': EXPERIMENT_MODE, 'variant': LIP_VARIANT, 'run_tag': RUN_TAG,
    'source_repository': REPO_URL, 'requested_ref': GIT_REF, 'requested_commit': SOURCE_COMMIT,
    'source_commit': source_commit, 'dataset_dir_at_runtime': str(dataset_dir),
    'seed': cfg['seed'], 'resolved_config': runtime_cfg,
    'lip_implementation': {
        'reference_repository': 'https://github.com/sebgao/LIP',
        'reference_commit': '6724ec63c1f1ebefa25d7269e29d17c4411a7941',
        'reference_file': 'imagenet/lip_densenet.py',
        'width': 'B=C/4', 'geometry': 'k2/s2/p0', 'gate': '12*sigmoid',
        'instance_norm': {'eps': 1e-5, 'affine': True, 'track_running_stats': False,
                          'gamma_init': 1, 'beta_init': 0},
        'conv_bias': False, 'nonfinal_conv_init': 'kaiming_normal fan_in relu',
        'final_conv_init': 'zero', 'pooling_denominator_epsilon': None,
        'backbone_initialization': 'CPU RNG fork/restore around G construction; same common state as Control at the same seed',
        'sharing': False, 'custom_cuda': False,
    },
    'runtime_overrides': {'train.devices': DEVICES, 'val.devices': 1},
    'train_command': train_command, 'yaml_default_devices': cfg['train']['devices'],
    'batch_size_per_process': cfg['train']['batch_size'],
    'global_batch_size': cfg['train']['batch_size'] * DEVICES * cfg['train']['accumulate_grad_batches'],
    'gpu_names': [torch.cuda.get_device_name(i) for i in range(DEVICES)],
    'cuda_version': torch.version.cuda,
    'versions': {name: importlib.metadata.version(name) for name in
                 ('torch', 'torchvision', 'pytorch-lightning', 'torchmetrics', 'numpy', 'PyYAML')},
    'benchmark_scope': 'synthetic single-GPU Control vs selected LIP placement; see benchmark/benchmark.json',
    'offline_analysis_requested': RUN_OFFLINE_ANALYSIS, 'analysis_max_images': ANALYSIS_MAX_IMAGES,
    'status': 'completed',
}
(RUN_DIR / 'run_manifest.json').write_text(json.dumps(manifest, indent=2), encoding='utf-8')
required_outputs = [TRAIN_LOG_DIR / 'checkpoints/last.ckpt', TRAIN_LOG_DIR / 'val_history.csv',
                    RUN_DIR / 'benchmark/benchmark.json', runtime_cfg_path]
missing = [str(path) for path in required_outputs if not path.is_file()]
if missing:
    raise FileNotFoundError('Missing artifacts: ' + ', '.join(missing))
archive = shutil.make_archive(str(WORKING_DIR / f'{SCENARIO}_{LIP_VARIANT}_{RUN_TAG}_artifacts'),
                              'zip', root_dir=RUN_DIR)
print(f'[DONE] Download this Kaggle output: {archive}')
print(f'[DONE] Source commit: {source_commit}')
